In [1]:
import numpy as np
import keras
from keras.models import model_from_json
from sklearn.metrics import confusion_matrix, classification_report

def kl_divergence(p, q, eps=1e-10):
    p = np.array(p) + eps
    q = np.array(q) + eps
    return np.sum(p * np.log(p / q))

listik = ['dis', 'gio', 'neu', 'pau', 'rab', 'sor', 'tri']
num_classes = len(listik)

def get_prediction_distribution(y_pred_labels, num_classes):
    hist = np.bincount(y_pred_labels, minlength=num_classes)
    dist = hist / np.sum(hist)
    return dist

# ===== Directories =====
savedir_both = 'emovo_gender_both'
savedir_female = 'emovo_gender_female'
savedir_male = 'emovo_gender_male'

num_folds = 5

opt = keras.optimizers.RMSprop(learning_rate=0.00001, decay=1e-6)

accuracies = []
conf_matrices = []
fold_reports = []
fold_macro_f1 = []

kl_values = []

for i in range(num_folds):

    # ------------------------------
    # Load Both-gender model (Baseline P)
    # ------------------------------
    model_name = f"Model_{i}"

    with open(f"{savedir_both}/{model_name}.json", "r") as json_file:
        loaded_model = model_from_json(json_file.read())

    loaded_model.load_weights(f"{savedir_both}/{model_name}.h5")
    loaded_model.compile(loss='categorical_crossentropy',
                         optimizer=opt,
                         metrics=['accuracy'])

    X_test = np.load(f"{savedir_both}/X_test_fold{i}.npy")
    y_test = np.load(f"{savedir_both}/y_test_fold{i}.npy")

    # Prediction baseline distribution P
    y_pred = loaded_model.predict(X_test, verbose=0)
    y_pred_labels = np.argmax(y_pred, axis=1)

    P = get_prediction_distribution(y_pred_labels, num_classes)

    # Accuracy + Metrics
    score = loaded_model.evaluate(X_test, y_test, verbose=0)
    acc = score[1] * 100
    accuracies.append(acc)

    y_true_labels = np.argmax(y_test, axis=1)

    cm = confusion_matrix(y_true_labels, y_pred_labels)
    conf_matrices.append(cm)

    report_dict = classification_report(
        y_true_labels,
        y_pred_labels,
        target_names=listik,
        digits=4,
        output_dict=True
    )

    fold_reports.append(report_dict)
    fold_macro_f1.append(report_dict['macro avg']['f1-score'])

    # ------------------------------
    # Load Female + Male models (Gender-aware Q)
    # ------------------------------

    # Female predictions
    female_model_name = f"Model_{i}"

    with open(f"{savedir_female}/{female_model_name}.json", "r") as json_file:
        female_model = model_from_json(json_file.read())

    female_model.load_weights(f"{savedir_female}/{female_model_name}.h5")
    female_model.compile(loss='categorical_crossentropy',
                         optimizer=opt,
                         metrics=['accuracy'])

    X_test_f = np.load(f"{savedir_female}/X_test_fold{i}.npy")

    y_pred_f = female_model.predict(X_test_f, verbose=0)
    y_pred_f_labels = np.argmax(y_pred_f, axis=1)

    Q_f = get_prediction_distribution(y_pred_f_labels, num_classes)

    # Male predictions
    with open(f"{savedir_male}/{female_model_name}.json", "r") as json_file:
        male_model = model_from_json(json_file.read())

    male_model.load_weights(f"{savedir_male}/{female_model_name}.h5")
    male_model.compile(loss='categorical_crossentropy',
                       optimizer=opt,
                       metrics=['accuracy'])

    X_test_m = np.load(f"{savedir_male}/X_test_fold{i}.npy")

    y_pred_m = male_model.predict(X_test_m, verbose=0)
    y_pred_m_labels = np.argmax(y_pred_m, axis=1)

    Q_m = get_prediction_distribution(y_pred_m_labels, num_classes)

    # Gender-aware distribution = average female + male
    Q = (Q_f + Q_m) / 2

    # KL divergence P || Q
    kl_values.append(kl_divergence(P, Q))

    print(f"\nFold {i}: Accuracy = {acc:.2f}%")

# ===============================
# Summary Statistics
# ===============================

print("\nMean Accuracy: %.2f%%" % np.mean(accuracies))
print("SD Accuracy: %.2f" % np.std(accuracies))

print(f"\nKL Divergence: {np.mean(kl_values):.6f}")

2026-03-05 16:41:07.404347: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-05 16:41:07.404621: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-05 16:41:07.433260: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-05 16:41:08.130938: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation or


Fold 0: Accuracy = 69.49%

Fold 1: Accuracy = 69.49%

Fold 2: Accuracy = 75.42%

Fold 3: Accuracy = 76.07%

Fold 4: Accuracy = 70.09%

Mean Accuracy: 72.11%
SD Accuracy: 2.98

KL Divergence: 0.035601


In [3]:
import numpy as np
import keras
from keras.models import model_from_json
from sklearn.metrics import confusion_matrix, classification_report

def kl_divergence(p, q, eps=1e-10):
    p = np.array(p) + eps
    q = np.array(q) + eps
    return np.sum(p * np.log(p / q))

listik = ['dis', 'gio', 'neu', 'pau', 'rab', 'sor', 'tri']
num_classes = len(listik)

def get_prediction_distribution(y_pred_labels, num_classes):
    hist = np.bincount(y_pred_labels, minlength=num_classes)
    dist = hist / np.sum(hist)
    return dist

# ===== Directories =====
savedir_both = 'emovo_speaker_both'
savedir_female = 'emovo_speaker_female'
savedir_male = 'emovo_speaker_male'

num_folds = 3

opt = keras.optimizers.RMSprop(learning_rate=0.00001, decay=1e-6)

accuracies = []
conf_matrices = []
fold_reports = []
fold_macro_f1 = []

kl_values = []

for i in range(num_folds):

    # ------------------------------
    # Load Both-gender model (Baseline P)
    # ------------------------------
    model_name = f"Model_{i}"

    with open(f"{savedir_both}/{model_name}.json", "r") as json_file:
        loaded_model = model_from_json(json_file.read())

    loaded_model.load_weights(f"{savedir_both}/{model_name}.h5")
    loaded_model.compile(loss='categorical_crossentropy',
                         optimizer=opt,
                         metrics=['accuracy'])

    X_test = np.load(f"{savedir_both}/X_test_fold{i}.npy")
    y_test = np.load(f"{savedir_both}/y_test_fold{i}.npy")

    # Prediction baseline distribution P
    y_pred = loaded_model.predict(X_test, verbose=0)
    y_pred_labels = np.argmax(y_pred, axis=1)

    P = get_prediction_distribution(y_pred_labels, num_classes)

    # Accuracy + Metrics
    score = loaded_model.evaluate(X_test, y_test, verbose=0)
    acc = score[1] * 100
    accuracies.append(acc)

    y_true_labels = np.argmax(y_test, axis=1)

    cm = confusion_matrix(y_true_labels, y_pred_labels)
    conf_matrices.append(cm)

    report_dict = classification_report(
        y_true_labels,
        y_pred_labels,
        target_names=listik,
        digits=4,
        output_dict=True
    )

    fold_reports.append(report_dict)
    fold_macro_f1.append(report_dict['macro avg']['f1-score'])

    # ------------------------------
    # Load Female + Male models (Gender-aware Q)
    # ------------------------------

    # Female predictions
    female_model_name = f"Model_{i}"

    with open(f"{savedir_female}/{female_model_name}.json", "r") as json_file:
        female_model = model_from_json(json_file.read())

    female_model.load_weights(f"{savedir_female}/{female_model_name}.h5")
    female_model.compile(loss='categorical_crossentropy',
                         optimizer=opt,
                         metrics=['accuracy'])

    X_test_f = np.load(f"{savedir_female}/X_test_fold{i}.npy")

    y_pred_f = female_model.predict(X_test_f, verbose=0)
    y_pred_f_labels = np.argmax(y_pred_f, axis=1)

    Q_f = get_prediction_distribution(y_pred_f_labels, num_classes)

    # Male predictions
    with open(f"{savedir_male}/{female_model_name}.json", "r") as json_file:
        male_model = model_from_json(json_file.read())

    male_model.load_weights(f"{savedir_male}/{female_model_name}.h5")
    male_model.compile(loss='categorical_crossentropy',
                       optimizer=opt,
                       metrics=['accuracy'])

    X_test_m = np.load(f"{savedir_male}/X_test_fold{i}.npy")

    y_pred_m = male_model.predict(X_test_m, verbose=0)
    y_pred_m_labels = np.argmax(y_pred_m, axis=1)

    Q_m = get_prediction_distribution(y_pred_m_labels, num_classes)

    # Gender-aware distribution = average female + male
    Q = (Q_f + Q_m) / 2

    # KL divergence P || Q
    kl_values.append(kl_divergence(P, Q))

    print(f"\nFold {i}: Accuracy = {acc:.2f}%")

# ===============================
# Summary Statistics
# ===============================

print("\nMean Accuracy: %.2f%%" % np.mean(accuracies))
print("SD Accuracy: %.2f" % np.std(accuracies))

print(f"\nKL Divergence: {np.mean(kl_values):.6f}")

/home/aruay/Desktop/gender-aware-SER-main/.venv/lib/python3.12/site-packages/keras/src/optimizers/base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(



Fold 0: Accuracy = 28.57%

Fold 1: Accuracy = 29.59%

Fold 2: Accuracy = 44.90%

Mean Accuracy: 34.35%
SD Accuracy: 7.47

KL Divergence: 0.528313
